# Teste individual dos 30 \(\theta\): energia e bitstring

Este bloco deve ser executado **no final do notebook principal do VQE**, depois da criação de `VQE_BEST`, `PROBLEMS`, `THETA_COLUMNS`, `problem_oracle` e `evaluate_theta_shots`.

O Hamiltoniano e 29 parâmetros permanecem fixos. Apenas um parâmetro é variado por vez:

\[
\theta_j\in[-\pi,\pi),\qquad \theta_{i\ne j}=\theta_i^{\mathrm{ref}}.
\]

Para cada intervenção, o código mede:

- energia média;
- probabilidade da solução ótima `p_best`;
- bitstring dominante;
- frequência com que o bitstring dominante é ótimo.

O teste responde: **quanto a energia e o bitstring mudam quando somente um \(\theta_j\) é alterado?**

In [ ]:

# ============================================================
# 1. CONFIGURAÇÃO
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

PROBLEM_ID = 0       # problema original do Yahoo Finance
N_ANGLES = 21        # pontos em [-pi, pi)
N_REPEATS = 2        # reduz ruído dos 4.096 shots
TOP_N = 5            # quantidade de perfis individuais
AUDIT_SEED = 912731

OUT = Path("outputs_theta_individual_test")
FIG = OUT / "figures"
TAB = OUT / "tables"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

required = [
    "VQE_BEST", "THETA_COLUMNS", "PROBLEMS", "problem_oracle",
    "evaluate_theta_shots", "circular_wrap", "CFG",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Execute antes o notebook principal. Objetos ausentes: "
        + ", ".join(missing)
    )

if len(THETA_COLUMNS) != 30:
    raise ValueError(f"Esperava 30 theta; encontrei {len(THETA_COLUMNS)}.")

print("Problema:", PROBLEM_ID)
print("Avaliações previstas:", 30 * N_ANGLES * N_REPEATS)


## 2. Vetor de referência

O vetor de referência é o melhor \(\theta\) salvo pelo COBYLA para `problem_id = 0`. O Hamiltoniano desse problema permanece fixo durante todo o teste.

In [ ]:

# ============================================================
# 2. CARREGAMENTO DO MELHOR VETOR DO PROBLEMA FIXO
# ============================================================
rows = VQE_BEST.loc[
    VQE_BEST["problem_id"].astype(int) == PROBLEM_ID
].copy()
if rows.empty:
    raise RuntimeError(f"Não há solução salva para problem_id={PROBLEM_ID}.")

energy_columns = ["confirmed_shot_energy", "shot_energy", "best_shot_energy"]
energy_column = next((c for c in energy_columns if c in rows.columns), None)
if energy_column is not None:
    rows[energy_column] = pd.to_numeric(rows[energy_column], errors="coerce")
    reference_row = rows.sort_values(energy_column).iloc[0]
else:
    reference_row = rows.iloc[0]

theta_reference = circular_wrap(
    reference_row[THETA_COLUMNS].to_numpy(dtype=float)
)
Q_FIXED = PROBLEMS["Q"][PROBLEM_ID]
CONSTANT_FIXED = float(PROBLEMS["constant"][PROBLEM_ID])
ORACLE_FIXED = problem_oracle(PROBLEM_ID)
OPTIMAL_BITSTRINGS = {str(x) for x in ORACLE_FIXED["optimal_bitstrings"]}

print("Energia exata:", ORACLE_FIXED["minimum_energy"])
print("Bitstring(s) ótimo(s):", sorted(OPTIMAL_BITSTRINGS))
print("Vetor de referência:", theta_reference.shape)


## 3. Avaliação repetida

Cada vetor é medido com sementes diferentes. A energia e `p_best` são calculados pela média. A frequência `optimal_dominant_rate` indica quantas repetições produziram um bitstring dominante ótimo.

In [ ]:

# ============================================================
# 3. FUNÇÃO DE AVALIAÇÃO
# ============================================================
def evaluate_for_audit(theta, seed_group):
    energies, p_best, dominant = [], [], []

    for repeat in range(N_REPEATS):
        metrics = evaluate_theta_shots(
            circular_wrap(theta),
            Q_FIXED,
            CONSTANT_FIXED,
            ORACLE_FIXED,
            seed=AUDIT_SEED + 100000 * int(seed_group) + repeat,
        )
        energies.append(float(metrics["shot_energy"]))
        p_best.append(float(metrics["p_best"]))
        dominant.append(str(metrics["dominant_bitstring"]))

    dominant_mode = pd.Series(dominant, dtype="object").mode().iloc[0]
    return {
        "mean_energy": float(np.mean(energies)),
        "std_energy": float(np.std(energies)),
        "mean_p_best": float(np.mean(p_best)),
        "dominant_bitstring": str(dominant_mode),
        "optimal_dominant_rate": float(np.mean([
            bitstring in OPTIMAL_BITSTRINGS for bitstring in dominant
        ])),
    }

baseline = evaluate_for_audit(theta_reference, seed_group=0)
print("Energia da referência:", baseline["mean_energy"])
print("p_best da referência:", baseline["mean_p_best"])
print("Bitstring dominante da referência:", baseline["dominant_bitstring"])


## 4. Varredura dos 30 parâmetros

O código percorre os 30 parâmetros. Em cada execução, altera somente um elemento do vetor e salva o resultado completo.

In [ ]:

# ============================================================
# 4. VARREDURA UM-THETA-DE-CADA-VEZ
# ============================================================
angles = np.linspace(-np.pi, np.pi, N_ANGLES, endpoint=False)
records = []

with tqdm(total=30 * N_ANGLES, desc="Varrendo theta") as bar:
    for theta_index in range(30):
        for angle_index, angle in enumerate(angles):
            candidate = theta_reference.copy()
            candidate[theta_index] = angle

            result = evaluate_for_audit(
                candidate,
                seed_group=1 + theta_index * N_ANGLES + angle_index,
            )

            records.append({
                "theta_index": theta_index,
                "theta_name": THETA_COLUMNS[theta_index],
                "angle_rad": float(angle),
                "angle_deg": float(np.degrees(angle)),
                "reference_angle_rad": float(theta_reference[theta_index]),
                "mean_energy": result["mean_energy"],
                "std_energy": result["std_energy"],
                "delta_energy": result["mean_energy"] - baseline["mean_energy"],
                "mean_p_best": result["mean_p_best"],
                "delta_p_best": result["mean_p_best"] - baseline["mean_p_best"],
                "dominant_bitstring": result["dominant_bitstring"],
                "optimal_dominant_rate": result["optimal_dominant_rate"],
                "same_reference_bitstring": (
                    result["dominant_bitstring"] == baseline["dominant_bitstring"]
                ),
            })
            bar.update(1)

sweep = pd.DataFrame(records)
sweep.to_csv(TAB / f"theta_sweep_problem_{PROBLEM_ID:03d}.csv", index=False)
print("Tabela completa:", sweep.shape)
sweep.head()


## 5. Resumo dos parâmetros

A `energy_span` mede a faixa total de energia causada pelo parâmetro:

\[
E_{\max}-E_{\min}.
\]

A `bitstring_flip_rate` mede a fração de ângulos que trocaram o bitstring dominante em relação ao vetor de referência.

In [ ]:

# ============================================================
# 5. RESUMO DOS 30 THETA
# ============================================================
summary = []
for theta_index, group in sweep.groupby("theta_index", sort=True):
    summary.append({
        "theta_index": int(theta_index),
        "theta_name": group["theta_name"].iloc[0],
        "energy_span": float(group["mean_energy"].max() - group["mean_energy"].min()),
        "maximum_energy_worsening": float(group["delta_energy"].max()),
        "p_best_span": float(group["mean_p_best"].max() - group["mean_p_best"].min()),
        "bitstring_flip_rate": float(1.0 - group["same_reference_bitstring"].mean()),
        "optimal_dominant_rate": float(group["optimal_dominant_rate"].mean()),
        "n_dominant_bitstrings": int(group["dominant_bitstring"].nunique()),
    })

summary = pd.DataFrame(summary)
energy_limit = summary["energy_span"].quantile(0.75)
flip_limit = summary["bitstring_flip_rate"].quantile(0.75)

def interpret(row):
    energy_high = row["energy_span"] >= energy_limit
    flip_high = row["bitstring_flip_rate"] >= flip_limit
    if energy_high and flip_high:
        return "modifica energia e bitstring"
    if energy_high:
        return "modifica principalmente a energia"
    if flip_high:
        return "modifica principalmente o bitstring"
    return "pouco sensível nesta referência"

summary["automatic_interpretation"] = summary.apply(interpret, axis=1)
summary = summary.sort_values("energy_span", ascending=False).reset_index(drop=True)
summary.to_csv(TAB / f"theta_summary_problem_{PROBLEM_ID:03d}.csv", index=False)
summary


# 6. Gráficos autocomentados

Os gráficos abaixo apresentam a mensagem principal dentro da própria figura.

In [ ]:

# ============================================================
# 6A. RANKING DA ENERGIA
# ============================================================
rank = summary.sort_values("energy_span")
fig, ax = plt.subplots(figsize=(12, 9))
ax.barh(rank["theta_name"], rank["energy_span"])
ax.set_xlabel("Faixa da energia ao variar somente um theta")
ax.set_ylabel("Parâmetro")
ax.set_title(f"Quais theta mais modificam a energia? — problema {PROBLEM_ID}")
ax.grid(axis="x", alpha=0.25)

top = summary.head(TOP_N)
text = ["LEITURA AUTOMÁTICA", "Maiores respostas:"]
text += [f"{r.theta_name}: {r.energy_span:.5f}" for r in top.itertuples()]
ax.text(0.98, 0.04, "\n".join(text), transform=ax.transAxes,
        ha="right", va="bottom",
        bbox={"boxstyle": "round,pad=0.5", "facecolor": "white", "alpha": 0.9})
fig.tight_layout()
fig.savefig(FIG / "01_energy_ranking.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:

# ============================================================
# 6B. RANKING DA TROCA DO BITSTRING
# ============================================================
rank = summary.sort_values("bitstring_flip_rate")
fig, ax = plt.subplots(figsize=(12, 9))
ax.barh(rank["theta_name"], 100 * rank["bitstring_flip_rate"])
ax.set_xlabel("Ângulos que trocaram o bitstring dominante (%)")
ax.set_ylabel("Parâmetro")
ax.set_title(
    "Quais theta mais modificam o bitstring?\n"
    f"Referência: {baseline['dominant_bitstring']}"
)
ax.grid(axis="x", alpha=0.25)

top = summary.sort_values("bitstring_flip_rate", ascending=False).head(TOP_N)
text = ["LEITURA AUTOMÁTICA", "Maiores taxas de troca:"]
text += [f"{r.theta_name}: {100*r.bitstring_flip_rate:.1f}%" for r in top.itertuples()]
ax.text(0.98, 0.04, "\n".join(text), transform=ax.transAxes,
        ha="right", va="bottom",
        bbox={"boxstyle": "round,pad=0.5", "facecolor": "white", "alpha": 0.9})
fig.tight_layout()
fig.savefig(FIG / "02_bitstring_ranking.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:

# ============================================================
# 6C. MAPA COMPLETO DA ENERGIA
# ============================================================
matrix = sweep.pivot(index="theta_index", columns="angle_rad", values="delta_energy").sort_index()
fig, ax = plt.subplots(figsize=(14, 9))
image = ax.imshow(
    matrix.to_numpy(), aspect="auto", origin="lower",
    extent=[-np.pi, np.pi, -0.5, 29.5],
)
ax.set_xlabel("Ângulo testado")
ax.set_ylabel("Índice do theta")
ax.set_title("Mudança da energia ao variar cada theta individualmente")
ax.set_xticks(
    [-np.pi, -np.pi/2, 0, np.pi/2, np.pi],
    [r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"],
)
ax.set_yticks(np.arange(30))
cb = fig.colorbar(image, ax=ax)
cb.set_label("Energia testada − energia da referência")

most = summary.iloc[0]
least = summary.iloc[-1]
ax.text(1.01, 0.98,
        "LEITURA AUTOMÁTICA\n"
        f"Mais sensível: {most.theta_name}\nFaixa: {most.energy_span:.5f}\n\n"
        f"Menos sensível: {least.theta_name}\nFaixa: {least.energy_span:.5f}",
        transform=ax.transAxes, ha="left", va="top",
        bbox={"boxstyle": "round,pad=0.5", "facecolor": "white", "alpha": 0.9})
fig.tight_layout()
fig.savefig(FIG / "03_energy_heatmap.png", dpi=220, bbox_inches="tight")
plt.show()


In [ ]:

# ============================================================
# 6D. MAPA DA PRESERVAÇÃO DO BITSTRING ÓTIMO
# ============================================================
matrix = sweep.pivot(
    index="theta_index", columns="angle_rad", values="optimal_dominant_rate"
).sort_index()
fig, ax = plt.subplots(figsize=(14, 9))
image = ax.imshow(
    matrix.to_numpy(), aspect="auto", origin="lower", vmin=0, vmax=1,
    extent=[-np.pi, np.pi, -0.5, 29.5],
)
ax.set_xlabel("Ângulo testado")
ax.set_ylabel("Índice do theta")
ax.set_title("O bitstring dominante ótimo é preservado quando cada theta muda?")
ax.set_xticks(
    [-np.pi, -np.pi/2, 0, np.pi/2, np.pi],
    [r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"],
)
ax.set_yticks(np.arange(30))
cb = fig.colorbar(image, ax=ax)
cb.set_label("Frequência do bitstring dominante ótimo")

best = summary.sort_values("optimal_dominant_rate", ascending=False).iloc[0]
worst = summary.sort_values("optimal_dominant_rate").iloc[0]
ax.text(1.01, 0.98,
        "LEITURA AUTOMÁTICA\n"
        f"Maior preservação: {best.theta_name}\n{100*best.optimal_dominant_rate:.1f}%\n\n"
        f"Menor preservação: {worst.theta_name}\n{100*worst.optimal_dominant_rate:.1f}%",
        transform=ax.transAxes, ha="left", va="top",
        bbox={"boxstyle": "round,pad=0.5", "facecolor": "white", "alpha": 0.9})
fig.tight_layout()
fig.savefig(FIG / "04_optimal_bitstring_heatmap.png", dpi=220, bbox_inches="tight")
plt.show()


## 7. Perfis individuais mais importantes

Para os parâmetros de maior resposta energética, cada figura mostra a energia em função do ângulo. Círculos indicam bitstring dominante ótimo; cruzes indicam outro bitstring.

In [ ]:

# ============================================================
# 7. PERFIS INDIVIDUAIS DOS TOP THETA
# ============================================================
for theta_index in summary.head(TOP_N)["theta_index"].astype(int):
    group = sweep.loc[sweep["theta_index"] == theta_index].sort_values("angle_rad")
    optimal = group["optimal_dominant_rate"] > 0
    info = summary.loc[summary["theta_index"] == theta_index].iloc[0]
    best_row = group.loc[group["mean_energy"].idxmin()]

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(group["angle_rad"], group["mean_energy"], marker=".", label="Energia média")
    ax.fill_between(
        group["angle_rad"],
        group["mean_energy"] - group["std_energy"],
        group["mean_energy"] + group["std_energy"],
        alpha=0.2, label="Variação entre repetições",
    )
    ax.scatter(group.loc[optimal, "angle_rad"], group.loc[optimal, "mean_energy"],
               marker="o", s=70, label="Bitstring dominante ótimo")
    ax.scatter(group.loc[~optimal, "angle_rad"], group.loc[~optimal, "mean_energy"],
               marker="x", s=70, label="Outro bitstring dominante")
    ax.axhline(baseline["mean_energy"], linestyle="--", label="Energia da referência")
    ax.axvline(theta_reference[theta_index], linestyle=":", label="Ângulo da referência")
    ax.set_xlabel("Ângulo testado")
    ax.set_ylabel("Energia medida")
    ax.set_title(f"Efeito individual de theta_{theta_index:02d} — outros 29 fixos")
    ax.set_xticks(
        [-np.pi, -np.pi/2, 0, np.pi/2, np.pi],
        [r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"],
    )
    ax.grid(alpha=0.25)
    ax.legend()
    ax.text(0.98, 0.04,
            "LEITURA AUTOMÁTICA\n"
            f"Faixa energética: {info.energy_span:.5f}\n"
            f"Troca do bitstring: {100*info.bitstring_flip_rate:.1f}%\n"
            f"Bitstrings encontrados: {info.n_dominant_bitstrings}\n"
            f"Melhor ângulo: {best_row.angle_rad:.3f} rad\n"
            f"Interpretação: {info.automatic_interpretation}",
            transform=ax.transAxes, ha="right", va="bottom",
            bbox={"boxstyle": "round,pad=0.5", "facecolor": "white", "alpha": 0.9})
    fig.tight_layout()
    fig.savefig(FIG / f"05_profile_theta_{theta_index:02d}.png", dpi=220, bbox_inches="tight")
    plt.show()


# 8. Conclusão automática

A conclusão separa três ideias:

1. parâmetros que mais alteram a energia;
2. parâmetros que mais trocam o bitstring;
3. parâmetros pouco sensíveis **neste vetor de referência**.

Um parâmetro pouco sensível individualmente ainda pode participar de compensações com outros parâmetros.

In [ ]:

# ============================================================
# 8. RELATÓRIO FINAL
# ============================================================
top_energy = summary.head(5)["theta_name"].tolist()
top_bitstring = summary.sort_values(
    "bitstring_flip_rate", ascending=False
).head(5)["theta_name"].tolist()
stable = summary.sort_values(
    ["energy_span", "bitstring_flip_rate"]
).head(5)["theta_name"].tolist()

print("=" * 72)
print("CONCLUSÃO — IMPACTO INDIVIDUAL DOS THETA")
print("=" * 72)
print("Hamiltoniano fixo, problem_id =", PROBLEM_ID)
print("Bitstring da referência:", baseline["dominant_bitstring"])
print("Energia da referência:", baseline["mean_energy"])
print()
print("Mais influentes na energia:", ", ".join(top_energy))
print("Mais influentes no bitstring:", ", ".join(top_bitstring))
print("Menos sensíveis nesta referência:", ", ".join(stable))
print()
print(
    "A coincidência entre os dois primeiros grupos identifica os candidatos "
    "mais fortes a controlar simultaneamente a energia e a solução dominante."
)
print("Tabelas:", TAB.resolve())
print("Figuras:", FIG.resolve())

display(summary[[
    "theta_name", "energy_span", "bitstring_flip_rate",
    "optimal_dominant_rate", "n_dominant_bitstrings",
    "automatic_interpretation",
]])
